# 1. Set up your environment

In [1]:
!pip install langchain langchain-community chromadb transformers sentence-transformers bitsandbytes beautifulsoup4 unstructured

# 2. Crawl & load the website content

In [25]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    "https://en.wikipedia.org/wiki/Artificial_intelligence",
    "https://en.wikipedia.org/wiki/Machine_learning"
]

loader = UnstructuredURLLoader(urls=urls)
raw_docs = loader.load()

In [26]:
raw_docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence'}, page_content='Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Machine_learning'}, page_content='Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.')]

# 3. Split documents into manageable chunks

In [28]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(raw_docs)

# 4. Embed the chunks & build your ChromaDB vector store

In [29]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Use a lightweight HF embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
# Create (or connect to) a local Chroma collection
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="wiki_ai"
)

# 5. Load Llama 2 via HuggingFace and wrap as a LangChain LLM

In [33]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: y
Token is valid (permission: read).
The token `Read_Token` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate whe

In [39]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer
import transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


model_name = "meta-llama/Llama-2-7b-chat-hf"  # Official model
# model_name = "daryl149/llama-2-7b-chat-hf"   # Unofficial model



In [42]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [43]:
# 2. Wrap in a text-generation pipeline
pipe = pipeline("text-generation",
                model=model,
                tokenizer= tokenizer,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                max_new_tokens = 512,
                do_sample=True,
                top_k=30,
                num_return_sequences=1,
                eos_token_id=tokenizer.eos_token_id
                )


# 3. LangChain LLM
llm=HuggingFacePipeline(pipeline=pipe, model_kwargs={'temperature':0})

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'eos_token_id', 'top_k', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_1737/2466060733.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm=HuggingFacePipeline(pipeline=pipe, model_kwargs={'temperature':0})


In [45]:
llm.invoke("Please provide a concise summary of the Book Harry Potter")

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"Please provide a concise summary of the Book Harry Potter and the Philosopher's Stone by J.K. Rowling.\n\nHarry Potter is an orphan who lives with his cruel and neglectful relatives, the Dursleys. On his eleventh birthday, he discovers that he is a wizard and begins attending Hogwarts School of Witchcraft and Wizardry. There, he makes friends with Ron Weasley and Hermione Granger and becomes involved in a mystery surrounding the powerful Sorcerer's Stone.\n\nHarry learns that the stone is hidden at Hogwarts and that a dark wizard named Lord Voldemort is trying to return to power and steal the stone to become immortal. Harry, Ron, and Hermione must use their skills and resourcefulness to uncover the truth and stop Voldemort before he can achieve his goal.\n\nThroughout the book, Harry faces challenges and obstacles, including difficult classes, bullies, and the temptation of the stone itself. With the help of his friends and the guidance of his teachers, Harry learns the importance of 

# 6. Assemble the RetrievalQA chain

In [48]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever()

prompt = PromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [50]:
query = qa_chain.invoke("Tell me about wiki_ai?")
print(query)

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following context to answer the question:

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence'}, page_content='Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.'), Document(metadata={'source': 'https://en.wikipedia.org/wiki/Machine_learning'}, page_content='Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.')]

Question: Tell me about wiki_ai?

Answer: Based on the provided context, it seems that wiki_ai is a term related to artificial intelligence (AI) and machine learning (ML). The two documents you provided are both from Wikipedia, and they mention the term "wiki_ai" in the context of AI and ML.

From the first document, we can see that wiki_ai is mentioned in the context of respecting the robot policy on Wikipedia. The policy requires users to set a user-agent and respect the website's robot 